## 1. IMPORT

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.lines as mlines
import random
import logging

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

logging.getLogger('tensorflow').setLevel(logging.ERROR)
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

# Global style - konsisten dengan notebook LSTM+HHO
_C = dict(
    train   = '#A0AEC0',
    val_act = '#63B3ED',
    act     = '#378ADD',
    val_p   = '#EF9F27',
    test_p  = '#E24B4A',
    split   = '#1D9E75',
    loss_tr = '#378ADD',
    loss_va = '#EF9F27',
)
HORIZON_COLORS = ['#1D9E75', '#63B3ED', '#EF9F27', '#E24B4A', '#7F77DD']

plt.rcParams.update({
    'figure.facecolor' : '#FFFFFF',
    'axes.facecolor'   : '#FFFFFF',
    'axes.edgecolor'   : '#E2E8F0',
    'axes.linewidth'   : 0.8,
    'axes.grid'        : True,
    'grid.color'       : '#E2E8F0',
    'grid.linewidth'   : 0.6,
    'grid.alpha'       : 1.0,
    'xtick.color'      : '#888888',
    'ytick.color'      : '#888888',
    'xtick.labelsize'  : 9,
    'ytick.labelsize'  : 9,
    'font.family'      : 'sans-serif',
    'font.size'        : 10,
    'text.color'       : '#1A202C',
})

def _spine(ax):
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)

def _fmt_usd(ax):
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:,.0f}'))

def _fmt_date(ax):
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.setp(ax.get_xticklabels(), rotation=0, ha='center')

def _subtitle(ax, txt):
    ax.annotate(txt, xy=(0, 1.04), xycoords='axes fraction',
                fontsize=8.5, color='#718096')

## 2. LOAD & PREPROCESS

In [ ]:
df = pd.read_csv('nvda_gabungan.csv')

if df.isnull().values.any():
    df = df.dropna()
    print("Dropped rows with missing values.")

df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df = df.sort_index()

TARGET_COLS = ['Open', 'High', 'Low', 'Close', 'Volume']
N_FEATURES  = len(TARGET_COLS)
data        = df[TARGET_COLS].copy()

print(f"Data shape : {data.shape}")
print(f"Periode    : {data.index[0].date()} -> {data.index[-1].date()}")

## 3. WALK-FORWARD SPLIT (70 / 15 / 15)

> Split dilakukan **sebelum** scaling, mengikuti split time-ordered yang sama
> dengan notebook LSTM MIMO + HHO, agar perbandingan kedua model adil (BAB 3.6.2).

In [ ]:
test_size = int(len(data) * 0.15)
val_size  = int(len(data) * 0.15)

test_df  = data.iloc[-test_size:]
val_df   = data.iloc[-(test_size + val_size):-test_size]
train_df = data.iloc[:-(test_size + val_size)]

print("=" * 65)
print("  WALK-FORWARD SPLIT (70/15/15) - LSTM MIMO Standar (Baseline)")
print("=" * 65)
for label, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"  {label:<6} | {d.index[0].date()} -> {d.index[-1].date()} "
          f"| {len(d):>4} baris | Close min={d['Close'].min():.2f}  max={d['Close'].max():.2f}")

## 4. NORMALISASI (Min-Max per Fitur, fit hanya pada Train)

Sesuai BAB 3.4.2: seluruh variabel input (Open, High, Low, Close, Volume) dinormalisasi
langsung menggunakan Min-Max Normalization ke rentang [0, 1]. Scaler di-fit **hanya**
pada data training untuk menghindari data leakage, lalu digunakan untuk transform
val/test.

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(train_df)

train_scaled = scaler.transform(train_df)
val_scaled   = scaler.transform(val_df)
test_scaled  = scaler.transform(test_df)

close_idx = TARGET_COLS.index('Close')
print("Close price - scaler range (fit pada train):")
print(f"  min : {scaler.data_min_[close_idx]:.4f}")
print(f"  max : {scaler.data_max_[close_idx]:.4f}")

## 5. PEMBUATAN DATASET MIMO

In [ ]:
n_input    = 60   # window size (fixed, sesuai baseline default BAB 3.5.1)
n_forecast = 5    # forecast horizon (fixed, sesuai baseline default)

def create_dataset(dataset, n_input, n_forecast):
    """
    Strategi MIMO: X = window n_input hari x 5 fitur, y = Close n_forecast hari ke depan.
    """
    X, Y = [], []
    for i in range(n_input, len(dataset) - n_forecast):
        X.append(dataset[i - n_input:i, :])
        Y.append(dataset[i:i + n_forecast, close_idx])
    return np.array(X), np.array(Y)

X_train, y_train = create_dataset(train_scaled, n_input, n_forecast)
X_val,   y_val   = create_dataset(val_scaled,   n_input, n_forecast)
X_test,  y_test  = create_dataset(test_scaled,  n_input, n_forecast)

print(f"X_train : {X_train.shape}  -> (samples, n_input={n_input}, N_FEATURES={N_FEATURES})")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}  -> (samples, n_forecast={n_forecast}) [skala 0-1]")

## 6. GROUND TRUTH HARGA ABSOLUT (Inverse Transform Target)

In [ ]:
def inverse_close(scaled_values):
    """Inverse-transform kolom Close dari skala [0,1] -> harga USD asli."""
    samples, horizon = scaled_values.shape
    dummy = np.zeros((samples * horizon, N_FEATURES))
    dummy[:, close_idx] = scaled_values.reshape(-1)
    inv = scaler.inverse_transform(dummy)
    return inv[:, close_idx].reshape(samples, horizon)

y_train_abs = inverse_close(y_train)
y_val_abs   = inverse_close(y_val)
y_test_abs  = inverse_close(y_test)

print(f"y_train_abs : {y_train_abs.shape}  -> harga aktual dalam USD")
print(f"y_val_abs   : {y_val_abs.shape}")
print(f"y_test_abs  : {y_test_abs.shape}")
print(f"\nSample y_test_abs terakhir : {y_test_abs[-1].round(2)}")

## 7. ARSITEKTUR MODEL LSTM MIMO STANDAR (Baseline)

Hyperparameter default sesuai BAB 3.5.1 (Tabel Arsitektur Model LSTM) — **tidak**
dioptimasi (ini adalah model baseline, dibandingkan dengan LSTM+HHO pada notebook lain).

| Hyperparameter        | Nilai Default |
|---|---|
| LSTM Layer 1 units    | 128 |
| LSTM Layer 2 units    | 64  |
| Dropout Rate          | 0.2 |
| Optimizer             | Adam |
| Learning rate         | 0.001 |
| Batch Size            | 32 |

In [ ]:
UNITS_1        = 128
UNITS_2        = 64
DROPOUT_RATE   = 0.2
LEARNING_RATE  = 0.001
BATCH_SIZE     = 32
EPOCHS         = 100
PATIENCE       = 10

def build_lstm_mimo_baseline():
    model = Sequential([
        LSTM(UNITS_1, return_sequences=True, input_shape=(n_input, N_FEATURES)),
        Dropout(DROPOUT_RATE),
        LSTM(UNITS_2, return_sequences=False),
        Dropout(DROPOUT_RATE),
        Dense(n_forecast)
    ])
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='mse')
    return model

model = build_lstm_mimo_baseline()
model.summary()

## 8. TRAINING MODEL

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=PATIENCE,
                           restore_best_weights=True, verbose=0)

print("\nTraining LSTM MIMO Standar (baseline, hyperparameter default)...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    shuffle=False, callbacks=[early_stop], verbose=1
)
print(f"\nTraining selesai pada epoch ke-{len(history.history['loss'])}")

## 9. PREDIKSI & INVERSE TRANSFORM

In [ ]:
train_pred_scaled = model.predict(X_train)
val_pred_scaled   = model.predict(X_val)
test_pred_scaled  = model.predict(X_test)

train_predict = inverse_close(train_pred_scaled)
val_predict   = inverse_close(val_pred_scaled)
test_predict  = inverse_close(test_pred_scaled)

print(f"train_predict : {train_predict.shape}  (dalam USD)")
print(f"val_predict   : {val_predict.shape}")
print(f"test_predict  : {test_predict.shape}")
print(f"\nSample test_predict terakhir : {test_predict[-1].round(2)}")
print(f"Ground truth terakhir        : {y_test_abs[-1].round(2)}")

## 10. EVALUASI (RMSE, MAE, MAPE per Horizon)

In [ ]:
def evaluate(y_true, y_pred):
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    return mse, rmse, mae, mape

for label, y_true_abs, y_pred_abs in [
    ("TRAIN", y_train_abs, train_predict),
    ("VAL",   y_val_abs,   val_predict),
    ("TEST",  y_test_abs,  test_predict),
]:
    print("\n" + "="*65)
    print(f"  EVALUASI {label} SET - LSTM MIMO Standar (Baseline)")
    print("="*65)
    for i in range(n_forecast):
        mse, rmse, mae, mape = evaluate(y_true_abs[:, i], y_pred_abs[:, i])
        print(f"  Horizon t+{i+1} (Hari ke-{i+1}) | "
              f"RMSE={rmse:8.4f}  MAE={mae:8.4f}  MAPE={mape:.2f}%")
    mse_a, rmse_a, mae_a, mape_a = evaluate(y_true_abs.flatten(), y_pred_abs.flatten())
    print(f"\n  Average (all horizons)   | "
          f"RMSE={rmse_a:8.4f}  MAE={mae_a:8.4f}  MAPE={mape_a:.2f}%")

test_dates = data.iloc[-(test_size):].index[n_input: n_input + len(test_predict)]
val_dates  = data.iloc[-(test_size+val_size):-test_size].index[n_input: n_input + len(val_predict)]

## 11. TABEL RINGKASAN EVALUASI

In [ ]:
print("\n" + "="*75)
print("  RINGKASAN EVALUASI - LSTM MIMO STANDAR (BASELINE)")
print("="*75)
print(f"  {'Set':<8} {'Horizon':<14} {'RMSE':>10} {'MAE':>10} {'MAPE':>10}")
print(f"  {'-'*54}")

for label, y_true_abs, y_pred_abs in [
    ("TRAIN", y_train_abs, train_predict),
    ("VAL",   y_val_abs,   val_predict),
    ("TEST",  y_test_abs,  test_predict),
]:
    for i in range(n_forecast):
        _, rmse, mae, mape = evaluate(y_true_abs[:, i], y_pred_abs[:, i])
        h = f"t+{i+1} (Hari ke-{i+1})"
        print(f"  {label:<8} {h:<14} {rmse:>10.4f} {mae:>10.4f} {mape:>9.2f}%")
    _, rmse_a, mae_a, mape_a = evaluate(y_true_abs.flatten(), y_pred_abs.flatten())
    print(f"  {label:<8} {'Average':<14} {rmse_a:>10.4f} {mae_a:>10.4f} {mape_a:>9.2f}%")
    print(f"  {'-'*54}")

## 12. PLOT - Training & Validation Loss

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
fig.subplots_adjust(top=0.86, bottom=0.12, left=0.09, right=0.97)

ax.plot(history.history['loss'],     color=_C['loss_tr'], lw=1.6, label='Train Loss')
ax.plot(history.history['val_loss'], color=_C['loss_va'], lw=1.6, label='Val Loss', ls='--')

_spine(ax)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.4f}'))
ax.set_ylabel('MSE Loss', fontsize=9, color='#718096', labelpad=8)
ax.set_xlabel('Epoch', fontsize=9, color='#888888')
ax.set_title('LSTM MIMO Standar (Baseline) - Training & Validation Loss',
             fontsize=12, fontweight='bold', color='#1A202C', loc='left', pad=14)
_subtitle(ax, f'Min-Max Normalization · Hyperparameter Default · Early Stopping (patience={PATIENCE})')

handles = [
    mlines.Line2D([], [], color=_C['loss_tr'], lw=1.6,          label='Train Loss'),
    mlines.Line2D([], [], color=_C['loss_va'], lw=1.6, ls='--', label='Val Loss'),
]
ax.legend(handles=handles, fontsize=8, frameon=False, ncol=2,
          loc='upper left', bbox_to_anchor=(0, -0.10))
plt.tight_layout(); plt.show()

## 13. PLOT - Aktual vs Prediksi (t+1)

In [ ]:
train_close_plot = train_df['Close']
val_close_plot   = val_df['Close']
test_close_plot  = test_df['Close']

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train_close_plot.index, train_close_plot,
        label='Train', color='silver', lw=0.8)
ax.plot(val_close_plot.index, val_close_plot,
        label='Val (Actual)', color='steelblue', lw=1.0)
ax.plot(test_close_plot.index, test_close_plot,
        label='Test (Actual)', color='royalblue', lw=1.2)
ax.plot(val_dates,  val_predict[:, 0],
        label='Val Pred t+1',  ls='--', color='orange', lw=1.0)
ax.plot(test_dates, test_predict[:, 0],
        label='Test Pred t+1', ls='--', color='red',    lw=1.2)
ax.axvline(val_df.index[0],  color='gray',  lw=1.2, ls=':', alpha=0.8, label='Train/Val Split')
ax.axvline(test_df.index[0], color='black', lw=1.2, ls=':', alpha=0.8, label='Val/Test Split')
ax.set_title('LSTM MIMO Standar (Baseline) - Close Price (t+1 Prediction)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Price (USD)'); ax.legend(fontsize=8, ncol=3)
ax.grid(alpha=0.3); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xlabel('Date'); plt.tight_layout(); plt.show()

## 14. PLOT - MIMO Forecast (Semua Horizon)

In [ ]:
last_seq      = test_scaled[-n_input:].reshape(1, n_input, N_FEATURES)
last_pred_raw = model.predict(last_seq)
last_pred_inv = inverse_close(last_pred_raw)[0]

last_date    = data.index[-1]
future_dates = pd.bdate_range(start=last_date, periods=n_forecast + 1)[1:]

lookback   = 60
hist_dates = test_close_plot.index[-lookback:]
hist_close = test_close_plot.iloc[-lookback:].values
last_price = float(data['Close'].iloc[-1])

fig2, ax2 = plt.subplots(figsize=(14, 5))
fig2.subplots_adjust(top=0.88, bottom=0.13, left=0.07, right=0.98)

ax2.axvspan(hist_dates[0], last_date, color=_C['act'],   alpha=0.04, zorder=0)
ax2.axvspan(last_date, future_dates[-1] + pd.tseries.offsets.BDay(1),
            color=_C['split'], alpha=0.06, zorder=0)
ax2.axvline(x=last_date, color=_C['split'], lw=1.4, ls='--', zorder=3)

ax2.plot(hist_dates, hist_close, color=_C['act'], lw=2.0,
         label='Aktual (60h terakhir)', zorder=5)
ax2.scatter([last_date], [last_price], color=_C['act'], s=50, zorder=6)

for i in range(n_forecast):
    x_line = [last_date] + list(future_dates[:i + 1])
    y_line = [last_price] + list(last_pred_inv[:i + 1])
    ax2.plot(x_line, y_line, ls='--', lw=1.5, color=HORIZON_COLORS[i],
             label=f'Pred t+{i+1}', zorder=4)
    ax2.scatter([future_dates[i]], [last_pred_inv[i]], color=HORIZON_COLORS[i], s=40, zorder=5)
    diff = last_pred_inv[i] - last_price
    sign = '+' if diff >= 0 else ''
    pct  = diff / last_price * 100
    ax2.annotate(f'${last_pred_inv[i]:.2f}\n({sign}{pct:.1f}%)',
                 xy=(future_dates[i], last_pred_inv[i]),
                 xytext=(0, 12), textcoords='offset points',
                 ha='center', fontsize=7.5, color=HORIZON_COLORS[i], fontweight='500')

_spine(ax2); _fmt_usd(ax2); _fmt_date(ax2)
ax2.set_ylabel('Price (USD)', fontsize=9, color='#718096', labelpad=8)
ax2.set_title(f'LSTM MIMO Standar (Baseline) - Next {n_forecast} Trading Days Forecast',
              fontsize=12, fontweight='bold', color='#1A202C', loc='left', pad=14)
_subtitle(ax2, f'Base: ${last_price:.2f} ({last_date.strftime("%d %b %Y")}) '
    '· MIMO paralel - satu forward pass, tanpa error accumulation')

handles2 = [mlines.Line2D([], [], color=_C['act'], lw=2.0, label='Aktual')]
for i in range(n_forecast):
    handles2.append(mlines.Line2D([], [], color=HORIZON_COLORS[i],
                                  lw=1.5, ls='--', marker='o',
                                  markersize=4, label=f'Pred t+{i+1}'))
ax2.legend(handles=handles2, fontsize=8, ncol=6, frameon=False,
           loc='upper left', bbox_to_anchor=(0, -0.09))
plt.tight_layout(); plt.show()

## 15. SIMPAN MODEL & METADATA

In [ ]:
import json, os

os.makedirs('models', exist_ok=True)
model.save('models/lstm_mimo_baseline.h5')

metadata = {
    "model_name": "LSTM MIMO Standar (Baseline)",
    "n_input": n_input,
    "n_forecast": n_forecast,
    "features": TARGET_COLS,
    "hyperparameters": {
        "units_1": UNITS_1, "units_2": UNITS_2,
        "dropout_rate": DROPOUT_RATE, "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE
    },
    "normalization": "MinMaxScaler per-fitur (fit pada train only)",
    "split": "70/15/15 (walk-forward, time-ordered)"
}
with open('models/lstm_mimo_baseline_meta.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Model dan metadata baseline tersimpan di folder 'models/'.")